# CrewAI + Snowflake MCP Server Integration

This notebook demonstrates how to use CrewAI to access Snowflake data via the Snowflake-managed MCP server.

**Run this notebook externally** (local Jupyter, Google Colab, etc.) — not inside Snowflake.

## Prerequisites
1. A Snowflake Programmatic Access Token (PAT)
2. The MCP server `CUSTOMER_DB.PROFILES.CUSTOMER_MCP_SERVER` deployed in Snowflake

In [1]:
# Stage 1: Install Dependencies
# Uncomment the line below if packages are not already installed
!pip install crewai crewai-tools


INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 9.6 MB/s  0:00:0236m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 8.5 MB/s  0:00:0536m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 14.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.6/803.6 kB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 10.2 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 6.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 8.5 MB/s  0:00:0036m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:

import importlib
for pkg in ['crewai', 'crewai_tools']:
    spec = importlib.util.find_spec(pkg)
    status = 'INSTALLED' if spec else 'NOT FOUND'
    print(f"{pkg}: {status}")

crewai: INSTALLED
crewai_tools: INSTALLED


In [3]:
# Stage 2: Configuration
import os

# Set credentials via environment variables (do not hardcode secrets in the notebook)
SNOWFLAKE_PAT = os.environ.get("SNOWFLAKE_PAT", "")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
SNOWFLAKE_ACCOUNT_URL = os.environ.get(
    "SNOWFLAKE_ACCOUNT_URL",
    "https://ngnwjus-mf49199.snowflakecomputing.com"
)

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# MCP Server endpoint
MCP_SERVER_URL = (
    f"{SNOWFLAKE_ACCOUNT_URL}/api/v2/databases/CUSTOMER_DB"
    f"/schemas/PROFILES/mcp-servers/CUSTOMER_MCP_SERVER"
)

print("Configuration:")
print(f"  Account URL: {SNOWFLAKE_ACCOUNT_URL}")
print(f"  MCP Server URL: {MCP_SERVER_URL}")
print(f"  PAT configured: {'Yes' if SNOWFLAKE_PAT else 'NO - export SNOWFLAKE_PAT'}")
print(f"  OpenAI key configured: {'Yes' if OPENAI_API_KEY else 'NO - export OPENAI_API_KEY'}")

Configuration:
  Account URL: https://ngnwjus-mf49199.snowflakecomputing.com
  MCP Server URL: https://ngnwjus-mf49199.snowflakecomputing.com/api/v2/databases/CUSTOMER_DB/schemas/PROFILES/mcp-servers/CUSTOMER_MCP_SERVER
  PAT configured: Yes


In [4]:
# Stage 3: Connect to MCP Server & Discover Tools
import time
import httpx
from crewai_tools import MCPServerAdapter

if not SNOWFLAKE_PAT:
    raise ValueError(
        "SNOWFLAKE_PAT is not set. Run: export SNOWFLAKE_PAT='your_token' "
        "then restart the kernel and re-run from Stage 2."
    )

print("Step 1/3: Quick MCP auth check (should finish in ~1-5s)...")
t0 = time.time()
resp = httpx.post(
    MCP_SERVER_URL,
    headers={
        "Authorization": f"Bearer {SNOWFLAKE_PAT}",
        "Content-Type": "application/json",
        "X-Snowflake-Authorization-Token-Type": "PROGRAMMATIC_ACCESS_TOKEN",
    },
    json={"jsonrpc": "2.0", "id": 1, "method": "tools/list"},
    timeout=15.0,
)
print(f"  HTTP {resp.status_code} in {time.time() - t0:.1f}s")
if resp.status_code != 200:
    raise RuntimeError(f"MCP preflight failed: {resp.text[:500]}")

print("Step 2/3: Loading CrewAI MCP adapter (may take ~15-20s on first import)...")
t1 = time.time()
adapter = MCPServerAdapter(
    {
        "url": MCP_SERVER_URL,
        "transport": "streamable-http",
        "headers": {
            "Authorization": f"Bearer {SNOWFLAKE_PAT}",
            "Content-Type": "application/json",
            "X-Snowflake-Authorization-Token-Type": "PROGRAMMATIC_ACCESS_TOKEN",
        },
    },
    connect_timeout=20,
)
print(f"  Adapter ready in {time.time() - t1:.1f}s")

print("Step 3/3: Reading discovered tools...")
mcp_tools = adapter.tools

print(f"Successfully connected to MCP Server!")
print(f"Discovered {len(mcp_tools)} tool(s):")
print("-" * 40)
for tool in mcp_tools:
    print(f"  Name: {tool.name}")
    print(f"  Description: {tool.description}")
    print()
print(f"Total Stage 3 time: {time.time() - t0:.1f}s")

TypeError: MCPServerAdapter.__init__() got an unexpected keyword argument 'url'

In [ ]:
# Stage 4: Create CrewAI Agent
from crewai import Agent

data_analyst = Agent(
    role="Airline Customer Data Analyst",
    goal="Query and analyze passenger profile data from Snowflake",
    backstory=(
        "You are a data analyst specializing in airline customer loyalty programs. "
        "You have access to a SQL execution tool that lets you query Snowflake. "
        "The passenger profiles are in CUSTOMER_DB.PROFILES.PASSENGER_PROFILES."
    ),
    tools=mcp_tools,
    verbose=True,
)

print("Agent created successfully!")
print(f"  Role: {data_analyst.role}")
print(f"  Tools available: {[t.name for t in data_analyst.tools]}")

In [ ]:
# Stage 5: Define Analysis Task
from crewai import Task

analysis_task = Task(
    description=(
        "Query the CUSTOMER_DB.PROFILES.PASSENGER_PROFILES table to: "
        "1. Count total passengers by loyalty tier. "
        "2. Find the average lifetime spend per tier. "
        "3. Identify which passengers have the highest retention propensity. "
        "Use the sql_exec_tool to run SQL queries."
    ),
    expected_output=(
        "A summary report showing passenger counts by tier, "
        "average spend per tier, and top passengers by retention propensity."
    ),
    agent=data_analyst,
)

print("Task defined:")
print(f"  Description: {analysis_task.description[:80]}...")
print(f"  Assigned to: {analysis_task.agent.role}")

In [ ]:
# Stage 6: Run the Crew & Get Results
from crewai import Crew

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY is not set. Run: export OPENAI_API_KEY='your_key' "
        "then restart the kernel and re-run from Stage 2."
    )

crew = Crew(
    agents=[data_analyst],
    tasks=[analysis_task],
    verbose=True,
)

print("Running CrewAI analysis...")
print("=" * 60)
try:
    # Jupyter already runs an asyncio event loop, so use async kickoff
    result = await crew.kickoff_async()
finally:
    adapter.stop()
print("=" * 60)
print("\nCrew execution complete!")

In [ ]:
# Stage 7: Display Final Results
print("=" * 60)
print("FINAL ANALYSIS REPORT")
print("=" * 60)
print()
print(result)